# Student AI MVP on Colab

最終更新日時: 2026-07-12 14:10:00 +09:00

一次方程式を学習する生徒AIをColabで実行するためのノートブックです。GitHubからこのリポジトリをcloneして、mock確認、パラメータ編集、対話授業、4bit LLM実行、ログ確認まで行います。

## 1. Clone repository

`REPO_URL` は設定済みです。すでに `/content/student-ai` が存在する場合はcloneをスキップします。

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_DIR = Path("/content/student-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /content/student-ai

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Optional Hugging Face login

Gemma系モデルやprivate/gated modelを使う場合に実行してください。

In [ ]:
# from huggingface_hub import login
# login()

## 4. Smoke test with mock model

モデルをダウンロードせず、状態管理・回答生成・ログ保存の経路を確認します。

In [ ]:
from src.student_ai import StudentAISimulator

sim = StudentAISimulator(use_mock_model=True)
result = sim.answer(
    student_id="S001",
    problem="2x + 3 = 11 を解いてください。",
)

print(result["answer"])

## 5. Model parameters

ここでモデルID、4bit量子化、生成パラメータを調整します。

- `MODEL_ID`: 使用するHugging FaceモデルID
- `load_in_4bit`: GPUメモリ節約。Colabでは基本 `True`
- `compute_dtype`: 計算精度。まず `bfloat16`、動かない場合は `float16`
- `max_new_tokens`: 生徒回答の最大長
- `temperature`: 高いほど回答が揺れる
- `top_p`: 低いほど候補語を絞る
- `repetition_penalty`: 同じ表現の繰り返しを抑える

In [ ]:
from src.config import GenerationConfig, ModelLoadConfig

# 使用するモデル。Gemma 3 4Bを使うなら "google/gemma-3-4b-it" などに変更します。
MODEL_ID = "Qwen/Qwen3-4B"

MODEL_LOAD_CONFIG = ModelLoadConfig(
    load_in_4bit=True,  # True: 4bit量子化でメモリ節約。Colabでは基本True
    bnb_4bit_quant_type="nf4",  # 通常はnf4
    bnb_4bit_use_double_quant=True,  # True: さらにメモリ節約
    compute_dtype="bfloat16",  # 動かないGPUではfloat16も試す
)

GENERATION_CONFIG = GenerationConfig(
    max_new_tokens=256,  # 長い返答が必要なら増やす。短い生徒回答なら128-256
    temperature=0.7,  # 低いほど安定、高いほど生徒らしい揺れが出る
    top_p=0.9,  # 低いほど保守的。通常は0.8-0.95
    do_sample=True,  # True: 毎回少し変わる。False: 安定寄り
    repetition_penalty=1.05,  # 同じ語句の繰り返し抑制
)

## 6. Student parameters

ここで使う生徒IDと、生徒AIの状態パラメータを編集します。実行すると `data/students/{STUDENT_ID}.json` に保存されます。

知識状態は `0-100` です。心理・性格系は `very_low / low / medium / high / very_high` の5段階です。

- `score`: 一次方程式全体の総合理解度
- `can_solve_ax_plus_b_equals_c`: `ax + b = c` 型を解く力
- `can_transpose_terms`: 移項の理解
- `can_divide_by_coefficient`: 係数で割る理解
- `can_handle_negative_numbers`: 負の数への対応
- `can_handle_fractions`: 分数への対応
- `misconceptions`: 誤概念。誤答や迷いとして発話に反映される
- `learning_speed`: 授業後の知識スコアの伸びやすさ
- `self_efficacy`: 自己効力感。自信の強さ
- `question_tendency`: 質問しやすさ
- `motivation`: 学習意欲
- `big_five`: 性格傾向。発話量、丁寧さ、不安の出方などに影響

In [ ]:
from src.state_manager import StateManager

STUDENT_ID = "S002"

state_manager = StateManager("data/students")
student_state = state_manager.load_student(STUDENT_ID)

# ここを編集すると、生徒AIの反応と、授業後の伸び方が変わります。
student_state["knowledge_state"] = {
    "linear_equation": {
        "level": "low",  # scoreから自動更新されます
        "score": 25,  # 一次方程式全体の理解度。0=未理解、100=十分理解
        "can_solve_ax_plus_b_equals_c": 25,  # ax + b = c 型を解く力
        "can_transpose_terms": 5,  # 移項で符号を変える理解
        "can_divide_by_coefficient": 20,  # 3x=15 で3で割る理解
        "can_handle_negative_numbers": 5,  # マイナスを含む計算への対応
        "can_handle_fractions": 5,  # 分数を含む式への対応
    }
}
student_state["misconceptions"] = [
    "移項は項を反対側へ動かすだけで符号はそのままだと思っている",  # 誤概念1
    "3x = 15 のような式で、3を引けばよいと考えることがある",  # 誤概念2
]
student_state["learning_speed"] = "low"  # 授業後の知識スコアの伸びやすさ
student_state["big_five"] = {
    "openness": "medium",  # 新しい説明や別解を受け入れやすいか
    "conscientiousness": "low",  # 丁寧に途中式を書くか
    "extraversion": "medium",  # 発話量や反応の積極性
    "agreeableness": "high",  # 教師に素直に合わせる傾向
    "neuroticism": "high",  # 不安や焦りの出やすさ
}
student_state["self_efficacy"] = "low"  # 自分は解けると思える度合い
student_state["question_tendency"] = "high"  # わからない時に質問する傾向
student_state["motivation"] = "medium"  # 学習意欲

state_manager.save_student(student_state)
student_state

## 7. Create simulator

授業と学力テストで使う生徒AIを作ります。実LLMで動かすため、標準では `USE_MOCK_MODEL=False` にしています。モデル読み込み前に軽く確認したい場合だけ `True` にしてください。

In [ ]:
from src.student_ai import StudentAISimulator

USE_MOCK_MODEL = False  # True: mockで軽く確認 / False: Qwenなどの実LLMで実行

sim = StudentAISimulator(
    model_id=MODEL_ID,
    load_in_4bit=MODEL_LOAD_CONFIG.load_in_4bit,
    model_load_config=MODEL_LOAD_CONFIG,
    generation_config=GENERATION_CONFIG,
    use_mock_model=USE_MOCK_MODEL,
)

print(f"student_id={STUDENT_ID}, model={sim.agent.model_id}")

## 8. Interactive lesson

教師として発話を入力すると、生徒AIが返答します。終了するには `exit` または `quit` と入力してください。

In [ ]:
while True:
    teacher_message = input("教師> ").strip()
    if teacher_message.lower() in {"exit", "quit", "終了"}:
        print("授業を終了します。")
        break
    if not teacher_message:
        continue

    result = sim.respond(STUDENT_ID, teacher_message)
    print("生徒AI>", result["answer"])
    print()

## 9. One-shot problem

対話ループを使わず、1問だけ出す場合はこちらを使います。

In [ ]:
result = sim.answer(
    student_id=STUDENT_ID,
    problem="3x - 5 = 10 を解いてください。",
)

print(result["answer"])

## 10. Controlled learning intervention

講義後に「この内容は学習できた」と明示的に反映するセルです。知識スコアを上げ、関連する誤概念を条件付きで解消します。

使い方:
- 講義前に `Understanding score vs accuracy graph` や `Assessment test` を実行
- `Interactive lesson` で授業をする
- このセルで学習できたスキルを上げる
- もう一度テストや相関グラフを実行して変化を見る

In [ ]:
# 講義で伸ばしたい分だけ指定します。値は加算量です。
# 例: 移項と係数で割る操作を重点的に教えた場合
learning_event = sim.apply_learning_intervention(
    STUDENT_ID,
    skill_deltas={
        "score": 15,
        "can_solve_ax_plus_b_equals_c": 15,
        "can_transpose_terms": 30,
        "can_divide_by_coefficient": 25,
        "can_handle_negative_numbers": 5,
        "can_handle_fractions": 0,
    },
    resolve_misconceptions=True,
)

print("学習介入結果:")
learning_event

## 11. Assessment test

生徒AIに学力テストを受けさせます。`Create simulator` で `USE_MOCK_MODEL=False` にしていれば、LLMが生徒として解答します。このテストは測定用なので、授業のように `knowledge_state` は更新しません。結果は `data/assessments/` に保存されます。

In [ ]:
from src.test_runner import TestRunner

TEST_ID = "linear_equation_basic_001"

runner = TestRunner(simulator=sim)
assessment_result = runner.run_test(
    student_id=STUDENT_ID,
    test_id=TEST_ID,
)

print("テスト:", assessment_result["title"])
print("得点:", assessment_result["score_percentage"], "%")
print("正答数:", assessment_result["correct_count"], "/", assessment_result["total_count"])
print("スキル別スコア:")
for skill, score in assessment_result["skill_scores"].items():
    print("-", skill, score, "%")

評価ログを確認します。

In [ ]:
from pathlib import Path

print(Path("data/assessments/human_readable.md").read_text(encoding="utf-8")[-2000:])

## 12. Understanding score vs accuracy graph

理解度スコアと20問テストの正答率の相関を確認します。理解度 `0,20,40,60,80,100` の検証用生徒を作り、同じ20問テストを受けさせて散布図を描きます。

- `TEST_ID_FOR_CORRELATION` は20問テストです。
- `update_knowledge=False` なので、テスト中に知識スコアは更新しません。
- テスト時は認知モデルが `knowledge_state` から正答/誤答方針を決め、LLMはその方針に沿って発話します。
- 講義前にこのセルを実行し、Interactive lessonで授業をした後にもう一度実行すると、授業前後の変化を比較できます。
- グラフに加えて、問題ごとの正誤表も表示・CSV保存します。
- LLMで確認する場合は `USE_MOCK_MODEL=False` の `sim` を使ってください。
- mock modelではパラメータを見ないため、相関確認には向きません。

In [ ]:
from copy import deepcopy
import math
import matplotlib.pyplot as plt
import pandas as pd

from src.state_manager import StateManager
from src.test_runner import TestRunner

TEST_ID_FOR_CORRELATION = "linear_equation_20q_001"
UNDERSTANDING_SCORES = [0, 20, 40, 60, 80, 100]

def score_to_level(score):
    if score < 20:
        return "very_low"
    if score < 40:
        return "low"
    if score < 60:
        return "medium"
    if score < 80:
        return "high"
    return "very_high"

state_manager = StateManager("data/students")
base_state = state_manager.load_student(STUDENT_ID)

runner = TestRunner(simulator=sim)
correlation_rows = []
answer_rows = []

for score in UNDERSTANDING_SCORES:
    profile_id = f"CORR_SCORE_{score:03d}"
    state = deepcopy(base_state)
    state["student_id"] = profile_id
    state["name"] = profile_id
    state["learning_history"] = []
    state["knowledge_state"] = {
        "linear_equation": {
            "level": score_to_level(score),
            "score": score,
            "can_solve_ax_plus_b_equals_c": score,
            "can_transpose_terms": score,
            "can_divide_by_coefficient": score,
            "can_handle_negative_numbers": score,
            "can_handle_fractions": score,
        }
    }
    state["misconceptions"] = [] if score >= 60 else [
        "移項しても符号は変えなくてよいと思っている",
        "係数で割る代わりに係数を引くことがある",
    ]
    state["self_efficacy"] = score_to_level(score)
    state["motivation"] = "medium"
    state["question_tendency"] = "high" if score < 40 else "medium"
    state_manager.save_student(state)

    result = runner.run_test(student_id=profile_id, test_id=TEST_ID_FOR_CORRELATION)
    correlation_rows.append({
        "student_id": profile_id,
        "understanding_score": score,
        "accuracy": result["score_percentage"],
        "correct_count": result["correct_count"],
        "total_count": result["total_count"],
    })
    for question_result in result["question_results"]:
        directive = question_result.get("assessment_directive", {})
        grading = question_result["grading"]
        answer_rows.append({
            "student_id": profile_id,
            "understanding_score": score,
            "question_id": question_result["question_id"],
            "skill": question_result["skill"],
            "difficulty": question_result["difficulty"],
            "expected_answer": question_result["expected_answer"],
            "student_value": grading["student_value"],
            "is_correct": grading["is_correct"],
            "mark": "○" if grading["is_correct"] else "×",
            "target_correct": directive.get("target_correct"),
            "correct_probability": directive.get("correct_probability"),
        })
    print(profile_id, "理解度=", score, "正答率=", result["score_percentage"], "%")

xs = [row["understanding_score"] for row in correlation_rows]
ys = [row["accuracy"] for row in correlation_rows]
x_mean = sum(xs) / len(xs)
y_mean = sum(ys) / len(ys)
numerator = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys))
den_x = math.sqrt(sum((x - x_mean) ** 2 for x in xs))
den_y = math.sqrt(sum((y - y_mean) ** 2 for y in ys))
correlation = numerator / (den_x * den_y) if den_x and den_y else 0

plt.figure(figsize=(7, 5))
plt.scatter(xs, ys)
plt.plot(xs, ys, linestyle="--", alpha=0.6)
for row in correlation_rows:
    plt.annotate(row["student_id"], (row["understanding_score"], row["accuracy"]), fontsize=8)
plt.title(f"Understanding score vs test accuracy (r={correlation:.2f})")
plt.xlabel("Knowledge score before test (0-100)")
plt.ylabel("Test accuracy (%)")
plt.xlim(-5, 105)
plt.ylim(-5, 105)
plt.grid(True, alpha=0.3)
plt.show()

correlation_df = pd.DataFrame(correlation_rows)
answer_df = pd.DataFrame(answer_rows)
correctness_table = answer_df.pivot(
    index="question_id",
    columns="understanding_score",
    values="mark",
)

correlation_df.to_csv("data/assessments/understanding_accuracy_summary.csv", index=False)
answer_df.to_csv("data/assessments/understanding_accuracy_detail.csv", index=False)
correctness_table.to_csv("data/assessments/understanding_accuracy_correctness_table.csv")

print("正誤表: 行=問題, 列=理解度スコア")
display(correctness_table)
print("詳細表")
display(answer_df)
print("CSV saved under data/assessments/")

correlation_df

## 13. Parameter error tendency check

パラメータによって誤答傾向が変わるかを確認します。同じ問題を、異なる生徒状態の検証用プロファイルに解かせ、回答と正誤を比較します。

- 元の `S001/S002/S003` は変更しません。
- 検証用に `PARAM_*` の生徒JSONを作ります。
- `update_knowledge=False` なので、この検証では知識スコアは更新しません。
- LLMで試す場合は `USE_MOCK_MODEL=False` の `sim` を先に作ってください。

In [ ]:
from copy import deepcopy
from src.grader import LinearEquationGrader
from src.state_manager import StateManager

CHECK_PROBLEM = "3x - 5 = 10 を解きなさい。"
EXPECTED_ANSWER = "x = 5"
TRIALS_PER_PROFILE = 3  # LLMの揺れを見るため、同じ条件で複数回試します

state_manager = StateManager("data/students")
base_state = state_manager.load_student("S002")
grader = LinearEquationGrader()

profiles = {
    "PARAM_LOW_KNOWLEDGE": {
        "description": "知識が低く、移項の誤概念が強い生徒",
        "knowledge_state": {
            "linear_equation": {
                "level": "very_low",
                "score": 10,
                "can_solve_ax_plus_b_equals_c": 15,
                "can_transpose_terms": 0,
                "can_divide_by_coefficient": 10,
                "can_handle_negative_numbers": 0,
                "can_handle_fractions": 0,
            }
        },
        "misconceptions": [
            "移項しても符号は変えなくてよいと思っている",
            "3x = 15 では3を引けばよいと思っている",
        ],
        "self_efficacy": "very_low",
        "question_tendency": "high",
        "motivation": "medium",
    },
    "PARAM_HIGH_ANXIETY": {
        "description": "知識は中程度だが不安が強く、迷いが出やすい生徒",
        "knowledge_state": {
            "linear_equation": {
                "level": "medium",
                "score": 55,
                "can_solve_ax_plus_b_equals_c": 60,
                "can_transpose_terms": 45,
                "can_divide_by_coefficient": 60,
                "can_handle_negative_numbers": 35,
                "can_handle_fractions": 25,
            }
        },
        "misconceptions": ["移項時の符号変化に自信がない"],
        "self_efficacy": "low",
        "question_tendency": "very_high",
        "motivation": "medium",
        "big_five": {
            "openness": "medium",
            "conscientiousness": "medium",
            "extraversion": "low",
            "agreeableness": "high",
            "neuroticism": "very_high",
        },
    },
    "PARAM_HIGH_KNOWLEDGE": {
        "description": "知識が高く、誤概念がほぼない生徒",
        "knowledge_state": {
            "linear_equation": {
                "level": "very_high",
                "score": 90,
                "can_solve_ax_plus_b_equals_c": 95,
                "can_transpose_terms": 90,
                "can_divide_by_coefficient": 90,
                "can_handle_negative_numbers": 85,
                "can_handle_fractions": 75,
            }
        },
        "misconceptions": [],
        "self_efficacy": "high",
        "question_tendency": "low",
        "motivation": "high",
    },
}

for profile_id, overrides in profiles.items():
    state = deepcopy(base_state)
    state["student_id"] = profile_id
    state["name"] = profile_id
    state["learning_history"] = []
    description = overrides.pop("description")
    state.update(overrides)
    state_manager.save_student(state)
    overrides["description"] = description

results = []
for profile_id, profile in profiles.items():
    for trial in range(1, TRIALS_PER_PROFILE + 1):
        response = sim.respond(profile_id, CHECK_PROBLEM, update_knowledge=False)
        grading = grader.grade(EXPECTED_ANSWER, response["answer"])
        results.append({
            "profile_id": profile_id,
            "description": profile["description"],
            "trial": trial,
            "is_correct": grading["is_correct"],
            "student_value": grading["student_value"],
            "answer": response["answer"],
        })

for item in results:
    mark = "OK" if item["is_correct"] else "NG"
    print(f"[{mark}] {item['profile_id']} trial={item['trial']} student_value={item['student_value']}")
    print("設定:", item["description"])
    print("回答:", item["answer"])
    print("-" * 80)

## 14. Check lesson logs

In [ ]:
from pathlib import Path

print(Path("data/logs/human_readable.md").read_text(encoding="utf-8")[-2000:])

## 15. Git update on Colab

GitHub側で更新したコードをColabへ取り込む場合は、次のセルを実行します。

In [ ]:
%cd /content/student-ai
!git status
!git pull

依存関係が変わった場合は、pull後に再インストールします。

In [ ]:
!pip install -q -r requirements.txt

pullで衝突したり、Colab上の作業内容を捨ててGitHubの最新版に戻したい場合は、次のセルでcloneし直します。必要な場合だけ実行してください。

In [ ]:
# 必要な場合だけ実行
# %cd /content
# !rm -rf student-ai
# !git clone {REPO_URL} /content/student-ai
# %cd /content/student-ai
# !pip install -q -r requirements.txt

## 16. Optional tests

標準テストはmock modelのみを使うため、LLMのダウンロードなしで実行できます。

In [ ]:
!python -m pytest